In [0]:
import pyspark.sql.functions as F

In [0]:
bronze_df = spark.read.table("meteo.bronze.raw_weather")

silver_df = (
    bronze_df
    .select(
        F.trim(F.col("city")).alias("city"),
        F.col("country"),
        F.col("latitude"),
        F.col("longitude"),
        F.to_timestamp("ingestion_time_utc").alias("ingestion_time_utc"),
        F.to_timestamp("api_response.current.time").alias("weather_time"),
        F.col("api_response.current.temperature_2m").alias("temperature_c"),
        F.col("api_response.current.relative_humidity_2m").alias("humidity_pct"),
        F.col("api_response.current.wind_speed_10m").alias("wind_speed_kmh"),
        F.col("api_response.current.precipitation").alias("precipitation_mm")
    )
    .withColumn("is_rainy", F.col("precipitation_mm") > 0)
    .withColumn("is_windy", F.col("wind_speed_kmh") > 30)
    .withColumn("is_cold", F.col("temperature_c") < 5)
)

silver_df.write.mode("append").format("delta").saveAsTable("meteo.silver.weather_silver")

In [0]:
%sql
select * from weather_silver